# 06 — Esboço do grafo heterogêneo

**O que vamos fazer aqui.** Consolidar o que os notebooks 01–05 mostraram em um esboço concreto do grafo que será construído na **Phase 1** e consumido pelo GNN na **Phase 2**. Sem código de modelo: só o **schema** (tipos de nó, tipos de aresta, features, janela temporal) e dois diagramas — um do schema abstrato e um exemplo real com vizinhança 1-hop de uma música do subset.

Este notebook é o entregável principal da pasta `exploration/`. Se algum colaborador pegar o repo daqui a um mês, ele deve ler **só este** para entender que grafo a gente está construindo e por quê.

## 1) Tipos de nó

Três tipos. Justificativa direta dos notebooks anteriores.

| Nó | # estimado | Origem | Justificativa |
|---|---|---|---|
| `music`  | ~5.000 (todo Top200) ou ~1.200 (subset alvo + 1-hop) | `data/songs/`  | Unidade de previsão. Carrega a trajetória de popularidade (target). |
| `artist` | ~1.700 | `data/artists/` | Hub natural — músicas do mesmo artista compartilham público; colaborações = ponte explícita (notebook 03). |
| `genre`  | ~530 | `data/MGDplus/genre_network/` | Categorias musicais com co-ocorrência estável ano a ano (notebook 04). Bom esqueleto semântico. |

**Decisão:** começar com esses três. *Não* incluir nós como `chart`, `country` ou `playlist` — sem ganho claro, custo de complexidade alto.

## 2) Tipos de aresta

Cinco tipos, todos justificáveis pelos achados.

| Aresta | Direção | Peso | Fonte |
|---|---|---|---|
| `artist —[performs]→ music` | dirigida | sem peso (ou 1/num_artists) | `songs.artist_id` (lista, multi-edge p/ colabs) |
| `music —[has_genre]→ genre` | dirigida | sem peso | derivada: gêneros do(s) artista(s) da música |
| `artist —[collaborates_with]— artist` | não-dirigida | `count` (# colabs) | `MGDplus/artist_network/br/` |
| `genre —[co_occurs]— genre` | não-dirigida | `Weight` (co-ocorrência) | `MGDplus/genre_network/br/` |
| `music —[co_trajectory]— music` | não-dirigida | similaridade (DTW ou correlação) | **a definir em Phase 1** |

**Decisão aberta:** a aresta `music↔music` por co-trajetória é a mais especulativa. Alternativas:
- Não incluir (mais simples; vizinhança via artista/gênero pode ser suficiente).
- Incluir só entre músicas que se sobrepõem temporalmente (mesma janela de chart).
- Incluir após threshold de similaridade (sparsificar).

Vamos começar **sem** essa aresta e adicionar como ablation se a Phase 2 ficar carente de sinal music-music.

## 3) Features por tipo de nó

Resumo do que veio dos notebooks 02 e 03.

| Nó | Features estáticas | Features dinâmicas (por janela) |
|---|---|---|
| `music` | 9 acústicas + `duration_ms`, `num_artists`, `explicit`, embedding(`release_year`) | trajetória `y` (do pré-processamento do paper) |
| `artist` | log(`num_hits`), log(`num_collab_hits`), fração colab, `years_on_charts` (multi-hot) | (opcional) popularidade agregada das músicas no chart na janela |
| `genre`  | embedding aprendido (sem features prontas) | (opcional) força agregada na janela |

## 4) Eixo temporal

**Decisão:** janela diária para o target (`y` da música) — bate com o pré-processamento do paper. Para a topologia do grafo, **redes anuais ou agregada total**, dada a estabilidade observada (notebook 04, Jaccard alto). Para a Phase 2, `HeteroGraphSAGE` codifica embeddings estruturais e um **GRU** processa a sequência temporal de embeddings × `y`.

In [ ]:
from pathlib import Path
import sys
import ast
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

ROOT = Path.cwd().resolve()
if ROOT.name == "exploration":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from music_diffusion_gnn.data.loaders import load_songs, load_artists
from music_diffusion_gnn.data.subset import load_subset

## 5) Diagrama do schema (abstrato)

Os três tipos de nó com as cinco arestas. Cores por tipo, traçado por relação.

In [ ]:
S = nx.MultiDiGraph()
S.add_node('music',  kind='music')
S.add_node('artist', kind='artist')
S.add_node('genre',  kind='genre')

S.add_edge('artist', 'music', label='performs')
S.add_edge('music',  'genre', label='has_genre')
S.add_edge('artist', 'artist', label='collaborates_with')
S.add_edge('genre',  'genre',  label='co_occurs')
S.add_edge('music',  'music',  label='co_trajectory (futuro)')

color_map = {'music': '#4C72B0', 'artist': '#DD8452', 'genre': '#55A868'}
pos = {'music': (0, 0), 'artist': (-1.4, 1.0), 'genre': (1.4, 1.0)}

fig, ax = plt.subplots(figsize=(8, 5.5))
nx.draw_networkx_nodes(
    S, pos, node_size=2400,
    node_color=[color_map[S.nodes[n]['kind']] for n in S.nodes()], ax=ax,
)
nx.draw_networkx_labels(S, pos, font_size=11, font_color='white', ax=ax)
for u, v, d in S.edges(data=True):
    style = 'arc3,rad=0.25' if u != v else 'arc3,rad=0.7'
    ax.annotate(
        '', xy=pos[v], xytext=pos[u],
        arrowprops=dict(arrowstyle='->' if u != v else '-|>',
                        connectionstyle=style, color='gray', lw=1.2,
                        shrinkA=22, shrinkB=22),
    )
    mid = ((pos[u][0] + pos[v][0]) / 2, (pos[u][1] + pos[v][1]) / 2 + 0.18)
    if u == v:
        mid = (pos[u][0] + 0.3, pos[u][1] - 0.5)
    ax.text(mid[0], mid[1], d['label'], fontsize=9, ha='center')
ax.set_title('Schema heterogêneo — 3 tipos de nó, 5 tipos de aresta')
ax.set_xlim(-2.5, 2.5); ax.set_ylim(-1.2, 1.8); ax.axis('off')
plt.tight_layout(); plt.show()

## 6) Exemplo concreto: vizinhança 1-hop de uma música do subset

Pegamos uma música real do subset, expandimos para os artistas que a executam, depois para os gêneros desses artistas, e finalmente conectamos colaboradores diretos. Isso é exatamente o tipo de subgrafo que o message passing 1-2 hops vai processar.

In [ ]:
songs   = load_songs()
artists = load_artists()
subset_ids = load_subset()
id_to_name = dict(zip(artists['artist_id'], artists['name']))

def parse_list(value):
    if isinstance(value, list): return value
    if isinstance(value, str) and value.startswith('['):
        try: return ast.literal_eval(value)
        except Exception: return []
    return []

# Selecionar uma música 'colaboração' do subset com bastante metadata
candidates = songs[
    songs['song_id'].isin(subset_ids)
    & (songs.get('song_type', '') == 'Collaboration')
    & (songs.get('num_artists', 0) >= 2)
]
if len(candidates) == 0:
    candidates = songs[songs['song_id'].isin(subset_ids)]
row = candidates.iloc[0]

song_id   = row['song_id']
song_name = row.get('song_name', song_id)
art_ids   = parse_list(row.get('artist_id', '[]'))
print(f'Música: {song_name}  ({song_id})')
print(f'Artistas: {[id_to_name.get(a, a) for a in art_ids]}')

# Gêneros via lookup nos artistas
genre_set = set()
art_genre = {}
for aid in art_ids:
    glist = artists.loc[artists['artist_id'] == aid, 'genres_list']
    if len(glist) > 0:
        gs = glist.iloc[0][:5]  # truncar a 5 p/ legibilidade
        art_genre[aid] = gs
        genre_set.update(gs)

print(f'Gêneros (via artistas, top 5 cada): {sorted(genre_set)}')

In [ ]:
# Construir o subgrafo de exemplo
G = nx.MultiDiGraph()
G.add_node(song_id, kind='music', label=song_name)
for aid in art_ids:
    G.add_node(aid, kind='artist', label=id_to_name.get(aid, aid))
    G.add_edge(aid, song_id, rel='performs')
for aid, gs in art_genre.items():
    for g in gs:
        G.add_node(g, kind='genre', label=g)
        G.add_edge(song_id, g, rel='has_genre')
# Colaboração entre os próprios artistas (se ≥2)
for i in range(len(art_ids)):
    for j in range(i + 1, len(art_ids)):
        G.add_edge(art_ids[i], art_ids[j], rel='collaborates_with')

color_map = {'music': '#4C72B0', 'artist': '#DD8452', 'genre': '#55A868'}
node_colors = [color_map[G.nodes[n]['kind']] for n in G.nodes()]
node_sizes = [1500 if G.nodes[n]['kind'] == 'music' else 900 for n in G.nodes()]
labels = {n: G.nodes[n].get('label', n)[:18] for n in G.nodes()}

pos = nx.spring_layout(G, seed=7, k=1.0)
fig, ax = plt.subplots(figsize=(11, 7))
nx.draw_networkx_edges(G, pos, alpha=0.4, ax=ax,
                       connectionstyle='arc3,rad=0.1')
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes,
                       alpha=0.9, ax=ax)
nx.draw_networkx_labels(G, pos, labels=labels, font_size=8, ax=ax)
ax.set_title(f'Vizinhança 1-hop de "{song_name}" (azul=music, laranja=artist, verde=genre)')
ax.axis('off')
plt.tight_layout(); plt.show()

## 7) Decisões — tomadas vs em aberto

**Tomadas (com base nos notebooks 01–05):**
1. Três tipos de nó: `music`, `artist`, `genre`.
2. Quatro tipos de aresta no v1: `performs`, `has_genre`, `collaborates_with`, `co_occurs`.
3. Granularidade temporal do target: **diária** (idem paper).
4. Topologia das redes (gênero, artista): **agregada do período** (estabilidade interanual confirmada).
5. Pré-processamento da trajetória `y`: rank → MA-7d → min-max [0, 0.5] → floor 0.001 (idem paper, já implementado em `preprocess.py`).
6. Conjunto-alvo = subset viral∩hit (1.179); conjunto-grafo ⊇ subset + 1-hop (artistas/gêneros associados).
7. Cobertura acústica = 100% no subset; sem necessidade de imputar.

**Em aberto (Phase 1 decide):**
1. Aresta `music↔music` por co-trajetória — incluir já ou só como ablation? Critério de similaridade?
2. Poda da rede de artistas (peso=1?) e da rede de gênero (limite mínimo de Weight?).
3. Direção das arestas `artist→music` e `music→genre` (vs não-dirigida c/ relação tipada): impacto no schema PyG.
4. `genre` ganha embedding aprendido ou usa one-hot? (ambos viáveis, decidir após Phase 1.)
5. Janela temporal do encoder GRU: 7d, 30d, 90d? Definir junto com a divisão treino/val/teste.
6. **Risco computacional:** message passing em grafo c/ ~5k músicas + ~1.7k artistas + ~500 gêneros é leve, mas se incluirmos `co_trajectory` com threshold baixo a aresta-set explode.

## 8) Próximos passos

Phase 1 (semanas 2–3 do roadmap):
- Materializar o grafo em **HeteroData** (`torch_geometric.data.HeteroData`).
- Persistir tensores de features e adjacências em `data/processed/graph_v1/`.
- Escrever testes de sanidade: nenhum nó isolado entre os targets, todos os artistas do subset alcançáveis a partir do conjunto música→artista, etc.
- Decidir as ambiguidades listadas em "Em aberto" via experimentos pequenos.